Genetic Algorithm

Helper Functions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def top_k(arr, k):
    return np.argpartition(arr, -k)[-k:]

def elitism(e, population, fitnesses):
    top = top_k(fitnesses, e)
    return population[top]

class Selector(object):
    def __init__(self):
        super().__init__()

    def prep_selector(self, fitnesses):
        pass

    def select(self, population):
        pass


class RouletteWheelSelector(Selector):
    def __init__(self):
        super().__init__()
        self.table = []

    def prep_selector(self, fitnesses):
        transformed = np.exp(fitnesses)
        total = np.sum(transformed)
        self.table = transformed / total
        self.indicies = np.arange(0, len(fitnesses), 1, dtype=int)

    def select(self, population):
        parent1_idx, parent2_idx = np.random.choice(self.indicies, size=2, replace=False, p=self.table)
        parent1 = population[parent1_idx, :]
        parent2 = population[parent2_idx, :]
        return parent1, parent2

class TournamentSelector(Selector):
    def __init__(self):
        super().__init__()
        self.table = []
    
    def prep_selector(self, fitnesses):
        self.table = fitnesses

    def get_winner(self,population):
        idx = np.random.choice(len(population), 5, replace=False)
        fitness = self.table[idx]
        winner = idx[np.argmax(fitness)]
        return winner
    
    def select(self,population):
        parent1 = self.get_winner(population)
        parent2 = self.get_winner(population)
        return parent1, parent2

def single_point(parent1, parent2):
    point = np.random.randint(1, len(parent1))
    child = np.zeros_like(parent1)
    child[:point] = parent1[:point]
    child[point:] = parent2[point:]
    return child


def interpolation(parent1, parent2):
    lambda_point = np.random.uniform()
    return lambda_point * parent1 +  (1 - lambda_point) * parent2

def uniform(parent1, parent2):
    selection = np.random.randint(0, 2, size=len(parent1))
    neg_selection = 1 - selection
    child = parent1 * selection + parent2 * neg_selection
    return child

#Ensures that crossing over dna returns an array with unique elements
def order_crossover(parent1, parent2):
    n = len(parent1)
    start = np.random.randint(0, n - 1)
    end = np.random.randint(start + 1, n)
    child = np.full(n, -1, dtype=int)
    child[start:end] = parent1[start:end]
    fill = [v for v in parent2 if v not in child[start:end]]
    idxs = [i for i in range(n) if child[i] == -1]
    for i, v in zip(idxs, fill):
        child[i] = v
    return child

class Game:
    def __init__(self, name, platform, genre, year, na, eu, jp, ww):
        self.name = name
        self.platform = platform
        self.genre = genre
        self.release = year
        self.nasales = na
        self.eusales = eu
        self.jpsales = jp
        self.wwsales = ww

class Company:
    def __init__(self, name):
        self.name = name
        self.sales2024 = 0
        self.sales2026 = 0
        self.games_2024 = []
        self.games_2026 = []

    def add_game(self, Game, year):
        total_sales = (
                Game.nasales +
                Game.eusales +
                Game.jpsales +
                Game.wwsales
            )
        
        if year == 2024:
            self.games_2024.append(Game)
            self.sales2024 += total_sales
        elif year == 2026:
            self.games_2026.append(Game)
            self.sales2026 += total_sales
    
    @property
    def profit(self):
        return self.sales2026 - self.sales2024
    
    @property
    def title_count(self):
        return len(self.games_2024)
    
    @property
    def genre_diversity(self):
        all_games = self.games_2024 + self.games_2026
        if not all_games:
            return 0.0
        genres = {g.genre for g in all_games}
        return len(genres) / len(all_games)
    
    @property
    def platform_count(self):
        all_games = self.games_2024
        return len({g.platform for g in all_games})
    
    @property
    def avg_sales_per_title(self):
        return self.total_sales / self.title_count if self.title_count > 0 else 0.0
    
    @property
    def total_sales(self):
        return self.sales2026
    
    def feature_array(self):
        total = self.total_sales if self.total_sales > 0 else 1e-9
        na = sum(g.nasales for g in self.games_2026)
        eu = sum(g.eusales for g in self.games_2026)
        jp = sum(g.jpsales for g in self.games_2026)
        ww = sum(g.wwsales for g in self.games_2026)
        
        return np.array([
            self.total_sales,          # 0 total_sales
            na / total,                # 1 na_ratio
            eu / total,                # 2 eu_ratio
            jp / total,                # 3 jp_ratio
            ww / total,                # 4 ww_ratio
            self.avg_sales_per_title,  # 5 avg_sales_title
            self.title_count,          # 6 title_count
            self.genre_diversity,      # 7 genre_diversity
            float(self.platform_count),# 8 platform_count
        ], dtype=float)
        '''
        return np.array([
            self.total_sales,          # 0 total_sales
            self.avg_sales_per_title,  # 1 avg_sales_title
            self.title_count,          # 2 title_count
            self.genre_diversity,      # 3 genre_diversity
            float(self.platform_count),# 4 platform_count
        ], dtype=float)'''
    
    def __repr__(self):
        return (
            f"Company(name={self.name}, "
            f"sales2024={self.sales2024:.2f}, "
            f"sales2026={self.sales2026:.2f}, "
            f"profit={self.profit:.2f})"
        )

def load_data(df, companies, year):
    for _, row in df.iterrows():
        publisher = row['Publisher']
    
        if publisher not in companies:
            companies[publisher] = Company(name=publisher)
        
        game = Game(
            row['Name'],
            row['Platform'],
            row['Genre'],
            row['Release_Year'],
            row['NA_Sales'],
            row['EU_Sales'],
            row['JP_Sales'],
            row['Other_Sales']
        )
    
        companies[publisher].add_game(game, year)

Random Data Sampling for testing

In [ ]:
df_2024 = pd.read_csv("../data/vgData2024.csv")
df_2026 = pd.read_csv("../data/vgData2026.csv")

all_companies = list(set(df_2024.Publisher.unique()) & set(df_2026.Publisher.unique()))

np.random.seed(250)
np.random.shuffle(all_companies)

split = int(0.75 * len(all_companies))
train_companies = set(all_companies[:split])
test_companies  = set(all_companies[split:])

companies = {}
load_data(df_2024, companies, year=2024)
load_data(df_2026, companies, year=2026)

for company in companies.values():
    if company.profit > 0:
        print(company)

np.random.seed(251)
train_sample = set(np.random.choice(list(train_companies), size=100, replace=False))

TrS_2024 = df_2024[df_2024.Publisher.isin(train_sample)]
TrS_2026 = df_2026[df_2026.Publisher.isin(train_sample)]

companiesTrS = {}
load_data(TrS_2024, companiesTrS, year=2024)
load_data(TrS_2026, companiesTrS, year=2026)

print("\n Training set \n")
for company in companiesTrS.values():
    if company.profit > 0:
        print(company)

np.random.seed(252)
test_sample = set(np.random.choice(list(test_companies), size=40, replace=False))

TS_2024 = df_2024[df_2024.Publisher.isin(test_sample)]
TS_2026 = df_2026[df_2026.Publisher.isin(test_sample)]

companiesTS = {}
load_data(TS_2024, companiesTS, year=2024)
load_data(TS_2026, companiesTS, year=2026)

print("\n Test set \n")
for company in companiesTS.values():
    if company.profit > 0:
        print(company)

Agent Code

In [ ]:

CDNA_FEATURES = [
    "total_sales",
    "na_market_ratio",
    "eu_market_ratio",
    "jp_market_ratio",
    "ww_market_ratio",
    "avg_sales_per_title",
    "title_count",
    "genre_diversity",
    "platform_count",
]
'''
CDNA_FEATURES = [
    "total_sales",
    "avg_sales_per_title",
    "title_count",
    "genre_diversity",
    "platform_count",
]'''
N_CDNA = len(CDNA_FEATURES)

class Agent:

    def __init__(self, cdna=None, dna=None, threshold=None, mutate_cdna=False,
        n_companies=10, companies=None, company_index=None):
        self.mutate_cdna = mutate_cdna
        self.n_companies = n_companies
        self._fitness = None
        
        self.companies = companies
        self.company_index = company_index
        
        if cdna is not None:
            self.cdna = np.array(cdna, dtype=float)
        else:
            self.cdna = np.random.uniform(0.0,1.0,N_CDNA)
            #self.cdna = np.random.randn(N_CDNA)
        
        if dna is not None:
            self.dna = np.array(dna, dtype=int)
        else:
            self.dna = np.random.permutation(n_companies)
            
        if threshold is not None:
            self.threshold = float(threshold)
        else:
            self.threshold = np.random.uniform(0.0,1.0)
            #self.threshold = np.random.uniform(0.1 * N_CDNA, 0.9 * N_CDNA)


    def score_company(self, company):
        fv = company.feature_array()
        scaled = _normalise_features(fv)
        #sig_cdna = sigmoid(self.cdna)
        return float(sigmoid(np.dot(scaled, self.cdna)))


    @property
    def fitness(self):
        return self._fitness
    
    @fitness.setter
    def fitness(self, value):
        self._fitness = float(value)
    
    @property
    def active_features(self):
        return [(CDNA_FEATURES[i], self.cdna[i]) for i, g in enumerate(self.cdna) if g > 0]
    
    @property
    def top_company(self):
        return int(self.dna[0])
    
    def classify(self, company):
        return int(self.score_company(company) >= self.threshold)
    
    @property
    def predictions(self):
        return {
            name: self.classify(self.companies[name])
            for name in self.company_index
        }
    
    def mutate(self, cdna_rate=0.05, dna_rate=0.1, threshold_rate = 0.05):
        new_cdna = self.cdna.copy()
        new_dna = self.dna.copy()
        new_threshold = self.threshold
        
        if self.mutate_cdna:
            mutation = np.random.random(N_CDNA) < cdna_rate
            rnd_noise = np.random.normal(0, 0.1, N_CDNA)
            new_cdna = np.where(mutation, new_cdna + rnd_noise, new_cdna)
            new_cdna = np.clip(new_cdna, 0.0, 1.0)
            if new_cdna.sum() == 0:
                new_cdna[np.random.randint(N_CDNA)] = np.random.uniform(0.1, 1.0)
        
        if np.random.random() < dna_rate:
            i, j = np.random.choice(self.n_companies, 2, replace=False)
            new_dna[i], new_dna[j] = new_dna[j], new_dna[i]

        if np.random.random() < threshold_rate:
            rnd_noise = np.random.normal(0, 0.05)
            new_threshold = np.clip(self.threshold + rnd_noise, 0.0, 1.0)
            #new_threshold = np.clip(self.threshold + rnd_noise, 0.0, float(N_CDNA))
        
        return Agent(new_cdna, new_dna, new_threshold, self.mutate_cdna, self.n_companies,
            self.companies, self.company_index)
    
    def crossover(self, other):
        mask = np.random.randint(0, 2, N_CDNA).astype(bool)
        c1 = np.where(mask,  self.cdna, other.cdna)
        c2 = np.where(mask, other.cdna,  self.cdna)
        for c in (c1, c2):
            if c.sum() == 0:
                c[np.random.randint(N_CDNA)] = np.random.uniform(0.1, 1.0)
        
        d1 = order_crossover(self.dna,  other.dna)
        d2 = order_crossover(other.dna, self.dna)

        t1 = (self.threshold + other.threshold) / 2
        t2 = t1 + np.random.normal(0.0, 0.1)
        t2 = np.clip(t2, 0.0, 1.0)
        #t2 = np.clip(t2, 0.0, float(N_CDNA))
        
        return (Agent(c1, d1, t1, self.mutate_cdna, self.n_companies,
            self.companies, self.company_index),
            Agent(c2, d2, t2, self.mutate_cdna, self.n_companies,
            self.companies, self.company_index))

    def __repr__(self):
        fit = f"{self._fitness:.4f}" if self._fitness is not None else "unevaluated"
        return (
            f"Agent(cdna={self.cdna.tolist()}, "
            f"active={self.active_features}, "
            f"top_co={self.top_company}, fitness={fit})"
        )

_feature_stats: dict = {}

def build_feature_stats(companies):
    vectors = np.array([c.feature_array() for c in companies.values()])
    _feature_stats['min'] = vectors.min(axis=0)
    _feature_stats['max'] = vectors.max(axis=0)

def _normalise_features(fv):
    if not _feature_stats:
        return fv
    mn, mx = _feature_stats['min'], _feature_stats['max']
    rng = np.where(mx - mn == 0, 1.0, mx - mn)
    return (fv - mn) / rng

def evaluate_fitness(agent, companies, company_index):
    n = len(company_index)
    correct = 0
    tp = tn = fp = fn = 0
    
    for i in range(n):
        company = companies[company_index[i]]
        predicted = agent.classify(company)
        actual = int(company.profit > 0)
        if predicted == actual:
            correct += 2 if actual == 1 else 1
        if predicted == 1 and actual == 1:
            tp += 1
        if predicted == 0 and actual == 0:
            tn += 1
        if predicted == 1 and actual == 0:
            fp += 1
        if predicted == 0 and actual == 1:
            fn += 1

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    balanced_accuracy = (sensitivity + specificity) / 2

    accuracy = correct / n
    
    #Bonus added if the Agent has a company in dna which has a positive profit value
    top_idx = agent.dna[0]
    top_company = companies[company_index[top_idx]]
    dna_bonus = (0.05 if top_company.profit > 0 else 0.0)
    
    #Bonus added if it predicted the company accurately
    pred_bonus = (0.05 if agent.classify(top_company) == int(top_company.profit > 0) else 0.0)

    #Bonus added if it predicted a company which is deemed profitable
    true_pos_bonus = (0.01 if agent.classify(top_company) == 1 
                      and top_company.profit > 0 else 0.0)
    
    #Penalty added if many features are used
    l1_penalty = -(0.01 * np.sum(np.abs(agent.cdna)))

    #Penalty added if weights are large
    l2_penalty = -(0.001 * np.sum(agent.cdna ** 2))

    #Penalty added if too many active features are used
    sparsity_penalty = -(0.02 * np.count_nonzero(agent.cdna > 0.1))
    
    #Bonus added if a company is relatively profitable compared to the highest profit company
    max_profit = max(companies[name].profit for name in company_index)
    profit_ratio = top_company.profit/max_profit
    profit_bonus = (0.05 * profit_ratio if top_company.profit > 0 else 0.0)

    return sigmoid(0.8 * balanced_accuracy + 0.1 * l1_penalty + 0.1 * profit_bonus) #+ 0.0 * sparsity_penalty) #+ l2_penalty  + true_pos_bonus  + dna_bonus + pred_bonus + profit_bonus

#Function to separate testing cell from Agent cell
def run_ga(
        companies,
        company_index,
        selector,
        pop_size,
        n_generations,
        elite_k,
        cdna_rate,
        dna_rate,
        mutate_cdna
    ):
    n_companies = len(company_index)
    
    population = [
        Agent(mutate_cdna=mutate_cdna, n_companies=n_companies,
        companies=companies, company_index=company_index)
        for _ in range(pop_size)
    ]
    
    fitness_history = []
    mean_fitness_history = []
    cdna_history = []
    best_agent = None

    mute_stagnation = 0
    best_curr_fitness = -np.inf
    current_cdna_rate = cdna_rate

    print("\nTraining Generations\n")
    
    for gen in range(n_generations):
    
        fitnesses = np.array([
            evaluate_fitness(agent, companies, company_index)
            for agent in population
        ])
        for agent, f in zip(population, fitnesses):
            agent.fitness = f
        
        best_idx = int(np.argmax(fitnesses))
        best_agent = population[best_idx]
        fitness_history.append(fitnesses[best_idx])
        mean_fitness_history.append(float(fitnesses.mean()))
        cdna_history.append(best_agent.cdna.copy())
        
        print(
            f"Gen {gen} \n best fitness: {fitnesses[best_idx]:.4f} "
            f"\n mean: {fitnesses.mean():.4f} "
            f"\n active feats: {best_agent.active_features}"
        )

        if fitnesses[best_idx] > best_curr_fitness + 1e-4:
            best_curr_fitness = fitnesses[best_idx]
            mute_stagnation = 0
            current_cdna_rate = cdna_rate
        else:
            mute_stagnation += 1
            if mute_stagnation > 10:
                current_cdna_rate = min(cdna_rate * 3, 0.3)
        
        elite_indices = top_k(fitnesses, elite_k)
        elites = [population[i] for i in elite_indices]
        
        pop_array = np.array([a.dna for a in population])
        selector.prep_selector(fitnesses)
        
        children = []
        while len(children) < pop_size - elite_k:
            p1_idx, p2_idx = selector.select(pop_array)
            p1 = population[p1_idx]
            p2 = population[p2_idx]
            c1, c2 = p1.crossover(p2)
            children.extend([c1.mutate(current_cdna_rate, dna_rate),
                c2.mutate(current_cdna_rate, dna_rate)])
        
        population = elites + children[:pop_size - elite_k]
    
    return best_agent, fitness_history, mean_fitness_history, cdna_history

Training Agents

In [ ]:
print(f"Companies loaded: {len(companiesTrS)}")

profitable_companies = 0;
for c in companiesTrS.values():
    if c.profit > 0:
        profitable_companies = profitable_companies + 1
        
print(f"Profitable companies: {profitable_companies}")

for c in companiesTrS.values():
    if c.profit > 0:
        print(f"  {c}")

company_index = list(companiesTrS.keys())

build_feature_stats(companies)

best, history, mean_history, cdna_history = run_ga(
    companies = companiesTrS,
    company_index = company_index,
    selector = TournamentSelector(),
    pop_size = 1000,
    n_generations = 100,
    elite_k = 50,
    cdna_rate = 0.05,
    dna_rate = 0.05,
    mutate_cdna = True
)

print(f"\nBest agent")
print(best)

top_name = company_index[best.top_company]
top_co = companiesTrS[top_name]
print(f"\nPredicted most profitable company : {top_name}")
print(f"  sales2024            = {top_co.sales2024:.2f}")
print(f"  sales2026            = {top_co.sales2026:.2f}")
print(f"  profit               = {top_co.profit:.4f}")
print(f"  is actually profitable: {top_co.profit > 0.0}")

print(f"\nTop 10 profitable companies:")
ranked = sorted(companiesTrS.values(), key=lambda c: c.profit, reverse=True)
for i, c in enumerate(ranked[:10], 1):
    marker = " <- predicted #1" if c.name == top_name else ""
    print(f"  {i}. {c}{marker}")

print(f"\nClassifications:")
preds = best.predictions
correct = sum(1 for name, pred in preds.items()
              if pred == int(companiesTrS[name].profit > 0))
print(f"  Accuracy: {correct}/{len(preds)} = {correct/len(preds):.1%}")
print(f"  {'Company':<40} {'Predicted':>10} {'Actual':>10} {'Match':>6}")
print(f"  {'-'*40} {'-'*10} {'-'*10} {'-'*6}")
for name, pred in preds.items():
    actual = int(companiesTrS[name].profit > 0)
    flag   = "Match" if pred == actual else "No Match"
    print(f"  {name:<40} {pred:>10} {actual:>10} {flag:>6}")

Testing Best Agent

In [ ]:
print(f"\nTest Set Classifications:")
print(f"  {'Company':<40} {'Predicted':>10} {'Actual':>10} {'Match':>6}")
print(f"  {'-'*40} {'-'*10} {'-'*10} {'-'*6}")

correct = 0
total = len(companiesTS)

for name, company in companiesTS.items():
    predicted = best.classify(company)
    actual = int(company.profit > 0)
    
    if predicted == actual:
        correct += 1
    
    flag = "Match" if predicted == actual else "No Match"
    print(f"  {name:<40} {predicted:>10} {actual:>10} {flag:>6}")

print(f"\nTest Set Accuracy: {correct}/{total} = {correct/total:.1%}")

Running Agent through entire dataset

In [ ]:
print(f"\nClassifying all Companies:")
print(f"  {'Company':<40} {'Predicted':>10} {'Actual':>10} {'Match':>6}")
print(f"  {'-'*40} {'-'*10} {'-'*10} {'-'*6}")

profitable_companies=[]

for name, company in companies.items():
    predicted = best.classify(company)
    actual = int(company.profit > 0)

    if(predicted == 1 and actual == 1):
        profitable_companies.append(company)
    
    flag = "Match" if predicted == actual else "No Match"
    print(f"  {name:<40} {predicted:>10} {actual:>10} {flag:>6}")

profitable_companies.sort(key=lambda c: c.profit, reverse=True)

print("\nCorrectly Predicted Profitable Companies (sorted by profit):")
for company in profitable_companies:
    print(company)

Plotting graphs to show fitness and dna changes

In [ ]:
def plot_fitness(fitness_history, mean_fitness_history):
    generations = range(len(fitness_history))
 
    fig, ax = plt.subplots(figsize=(10, 5))
 
    ax.plot(generations, fitness_history,
            label="Best fitness", color="#2563eb", linewidth=2)
    ax.plot(generations, mean_fitness_history,
            label="Mean fitness", color="#f97316", linewidth=2)
 
    ax.set_xlabel("Generation", fontsize=12)
    ax.set_ylabel("Fitness", fontsize=12)
    ax.set_title("Fitness over Generations", fontsize=14, fontweight="bold")
    ax.legend(fontsize=11)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.grid(True, linestyle=":", alpha=0.5)
    plt.tight_layout()
    plt.show()

def plot_cdna_history(cdna_history, feature_names=None):
    cdna_array = np.array(cdna_history).T
    n_features, n_gens = cdna_array.shape

    if feature_names is None:
        feature_names = [f"feature_{i}" for i in range(n_features)]

    generations = range(n_gens)
    cmap = plt.colormaps["tab10"].resampled(n_features)

    fig, ax = plt.subplots(figsize=(10, 5))

    for i, (feature, weights) in enumerate(zip(feature_names, cdna_array)):
        ax.plot(generations, weights, label=feature, color=cmap(i), linewidth=2)

    ax.set_xlabel("Generation", fontsize=12)
    ax.set_ylabel("CDNA Weight", fontsize=12)
    ax.set_title("Best Agent CDNA Weights over Generations", fontsize=14, fontweight="bold")
    ax.legend(fontsize=9, bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.grid(True, linestyle=":", alpha=0.5)
    plt.tight_layout()
    plt.show()

plot_fitness(history, mean_history)
plot_cdna_history(cdna_history, feature_names=CDNA_FEATURES)